# Day07_Advanced_Delta_SCD2

## **1. What is Delta Time Travel?**
        
Delta Time Travel lets you query an **earlier version of a Delta table** without changing the current version. You can use **DESCRIBE HISTORY to find a previous version** and then query it using **VERSION AS OF.**


## **2. Time Travel vs RESTORE?**

- **Time Travel** → reads/query a previous table version without modifying the current table.
- **RESTORE** → makes a previous version become the table's current state by creating a new Delta table version. 

## **3. What does VACUUM do?**

- removes **old/unreferenced data files** that are no longer required by the current table and are older than the retention threshold(>7days{default, unless changed}).
- The default retention is typically **7 days / 168 hours.**

## **4. Why can VACUUM affect Time Travel?**

Delta transaction history may still describe an old version, but Time Travel also needs the **underlying data files** referenced by that version. If VACUUM physically removes those files, that **old version may no longer be queryable.**

## **5. What does OPTIMIZE do?**

- **OPTIMIZE** -->  compacts many small files into fewer, larger files to** improve file layout **and **reduce** the number of files Spark needs to scan.

## **6. MERGE vs UPDATE — when would you use each?**

**update**--> updtaes the record based on teh condition. 

**Merge** --> 

    - compares source and target based on a merge condition.
    - can perform UPDATE, INSERT and DELETE depending on matched/not-matched conditions.
            -has methods like:
            ````
            .whenMatchedUpdateAll()
            .whenMatchedUpdate()
            .whenNotMatchedInsert()
            .whenNotMatchedInsertAll()
            ````

## **7. OPTIMIZE vs VACUUM?**

- **OPTIMIZE** -->  reorganizes files that are being kept
- **VACUUM** --> deletes obsolete files


## **8. What is SCD Type 1?**

- **overwrites** changed attributes and does not preserve history.
- doesnt store HISTORICAL DATA
- eg- 
    ```
    Before
    1001 | Plano

    After
    1001 | Frisco


## **9. What is SCD Type 2?**

- if there is a chnage in teh record it insert the record as new by decommissioning the old data/record by updating the iscurrentFlag as false, add the new record with iscurrent as true. so at any given time iscurrentFlad=True will fetch you the latest records

## **10. How would you implement SCD Type 2 using Delta?**

```
New customer
    → INSERT

Changed customer
    → expire current row
    → INSERT new current row

Unchanged customer
    → do nothing
```

**code**-->

````
from delta.tables import DeltaTable

upt_data = [
    (1001, "John Smith", "Suspended", "Dallas"),
    (1002, "Mary Jones", "Active", "Plano"),
    (1004, "Sarah Wilson", "Active", "Frisco")
]

dfUpdates = spark.createDataFrame(upt_data, scd_columns)

dfSCD2 = (
    dfUpdates
    .withColumn("startDate", current_date())
    .withColumn("endDate",lit(None).cast("date"))
    .withColumn("isCurrent",lit(True))
                )

# Get current ACTIVE records to compare for actual changes
dfCurrent = spark.table("workspace.customer360.customer_history").filter(col("isCurrent") == True)

# Only keep records that are new or have actual value changes
dfChanges = (
    dfSCD2.alias("new")
    .join(dfCurrent.alias("current"), "custId", "left")
    .filter(
        col("current.custId").isNull()
        |
        ~col("current.custName").eqNullSafe(col("new.custName"))
        |
        ~col("current.status").eqNullSafe(col("new.status"))
        |
        ~col("current.city").eqNullSafe(col("new.city"))
    )
    .select("new.*")
)

# deactive OLD record
currentMaster = DeltaTable.forName(spark,"workspace.customer360.customer_history")
(
currentMaster.alias("current").merge(
    dfChanges.alias("new"), "current.custId = new.custId AND current.isCurrent = true"
    ) 
    .whenMatchedUpdate(
        set = {
            "endDate": current_date(), "isCurrent": lit(False)
        }
    )
    .execute()
)

dfChanges.write.format("delta").mode("append").saveAsTable("workspace.customer360.customer_history")
````

## **11. What is an idempotent data pipeline?**
An idempotent pipeline can process the same input repeatedly without producing additional unintended changes after the first successful run.
consider above example- your current implementation is idempotent for this same input.

Suppose you run it a** second time**.

After run #1, current records are:
```
1001  Suspended   current
1002  Active      current
1003  Active      current
1004  Active      current
```

On run #2, your source contains:
```
1001  Suspended
1002  Active
1004  Active
```

Your comparison finds:
```
1001 → same
1002 → same
1004 → same
```
Therefore:
dfChanges = empty

So:
```
MERGE  → updates 0 rows
APPEND → inserts 0 rows
```


## **12. How do you make SCD2 pipeline idempotent?

Before expiring and inserting an SCD2 record, I compare the incoming attributes against the current target record. Only genuinely new or changed records enter dfChanges. Therefore, rerunning the same source produces an empty change set and does not generate duplicate history records.